# Exercise 5: Glot500 as a baseline

*Neural Morphology · version 0.1.0*

> Run TurkicNLP's `morph_neural` analyser on the IMST test set (read it with `turkicnlp.utils.conllu.read_conllu`, and process each sentence's text) and score UPOS and the full feature bundle against the gold annotation. How does a multilingual analyser serving 22 languages compare with the monolingual BERTurk tagger? On which features does each win?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" "transformers==5.3.0" sentencepiece protobuf scikit-learn stanza requests` (the compiled `hfst` package needs Python 3.11 or 3.12). Tokenisers are loaded from Hugging Face, or from a local copy in TurkicNLP's model folder (`~/.turkicnlp/models/huggingface/<org>--<name>`). Universal Dependencies files are read from the UD GitHub repositories (master branch), so internet access is needed. Cells whose model cannot be loaded print the reason instead of a result.

The cell runs the multilingual analyser sentence by sentence and scores only sentences whose automatic tokenisation matches the gold words. It needs the Glot500 backbone and the TurkicNLP multilingual morphology model, which are downloaded on first use. Full-bundle agreement with IMST also depends on annotation conventions: compare the core-bundle accuracy of Exercise 4 as well.

In [ ]:
import os, re, collections, statistics
from pathlib import Path
import requests

UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/master/{file}"
WORD = re.compile(r"[^\W_]+")                         # the book's word definition (Scripts and Encoding)

def ud_sentences(repo, file):
    """CoNLL-U sentences as lists of (form, lemma, upos, feats) for integer-ID words, with the sent_id."""
    text = requests.get(UD.format(repo=repo, file=file), timeout=120)
    text.raise_for_status()
    sents, words, sid = [], [], None
    for line in text.text.splitlines():
        if line.startswith("# sent_id"):
            sid = line.split("=", 1)[1].strip()
        elif not line.strip():
            if words:
                sents.append((sid, words))
            words = []
        elif not line.startswith("#"):
            c = line.split("\t")
            if c[0].isdigit():
                words.append((c[1], c[2], c[3], c[5]))
    if words:
        sents.append((sid, words))
    return sents

def load_tokenizer(name):
    """A Hugging Face tokenizer, from TurkicNLP's local model folder if present, otherwise from the Hub."""
    from transformers import AutoTokenizer
    local = Path("~/.turkicnlp/models/huggingface").expanduser() / name.replace("/", "--")
    return AutoTokenizer.from_pretrained(str(local) if (local / "tokenizer_config.json").exists() else name)

def pieces(tok, word):
    """Subword pieces of one word without special tokens and without the word-initial marker."""
    out = []
    for p in tok.tokenize(word):
        p = p.replace("▁", "").replace("Ġ", "")
        out.append(p[2:] if p.startswith("##") else p)
    return [p for p in out if p]

CORE_FEATS = ("Case", "Number", "Number[psor]", "Person", "Person[psor]", "Tense", "VerbForm", "Voice")
DEFAULTS = {("Case", "Nom"), ("Number", "Sing"), ("Person", "3")}

def core(upos, feats):
    """UPOS plus the core features, without the unmarked defaults Case=Nom, Number=Sing and Person=3,
    and with Polarity only if negative. Treebanks differ in whether they write the defaults out."""
    kv = dict(f.split("=", 1) for f in (feats or "").split("|") if "=" in f)
    keep = {k: v for k, v in kv.items() if k in CORE_FEATS and (k, v) not in DEFAULTS}
    if kv.get("Polarity") == "Neg":
        keep["Polarity"] = "Neg"
    return upos + "|" + "|".join(f"{k}={v}" for k, v in sorted(keep.items()))

In [ ]:
import turkicnlp
from turkicnlp import Pipeline
from turkicnlp.utils.conllu import read_conllu       # TurkicNLP's reader (the code below uses ud_sentences)

test = ud_sentences("UD_Turkish-IMST", "tr_imst-ud-test.conllu")
try:
    turkicnlp.download("tur", processors=["tokenize", "morph_neural"])
    neural = Pipeline("tur", processors=["tokenize", "morph_neural"], tokenize_backend="rule")
    upos_ok = feats_ok = total = 0
    for _, ws in test:
        pred = neural(" ".join(f for f, *_ in ws)).words
        if len(pred) != len(ws):
            continue                                      # tokenisation differs: skip, and count how often
        for (form, lemma, upos, feats), p in zip(ws, pred):
            total += 1; upos_ok += p.upos == upos; feats_ok += (p.feats or "_") == feats
    print(f"words {total}: UPOS {upos_ok/total:.3f}, full FEATS {feats_ok/total:.3f}")
except Exception as exc:
    print("Neural analyser not available here:", type(exc).__name__, str(exc)[:120])

## Worked solution

Output of `experiments/ch07_exercises/run_ch07_exercises.py` on 19 October 2026:

```
Neural analyser not available here: LocalEntryNotFoundError An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. 
```

The multilingual analyser could not be loaded in the environment used for the worked solution: it needs the Glot500 backbone from Hugging Face, which was not reachable, so the cell records the failure instead of scores. The comparison asked for also needs the BERTurk tagger of the Hands-On section, which is not trained in the book.

What can already be said is how to make the comparison fair. Score both systems on the same words, report the share of sentences whose tokenisation differs from the gold words, and score full FEATS *and* a convention-neutral core bundle, as in Exercise 4. Exercise 6 shows why: the Stanza model trained on IMST matches only 15.9% of the full feature bundles of the Turkish TueCL treebank, largely because the two Turkish treebanks write different features. A multilingual analyser trained on several treebanks will mix such conventions, and its full-bundle score against IMST will understate what it gets right.

## Your turn — extension

1. Run the cell where Hugging Face is reachable and report the share of sentences skipped because of tokenisation.
2. Score per feature (Case, Number, Person, Tense, VerbForm) and compare with the Stanza tagger of Exercise 4 on the same words.